## Silver to Gold Transformation

Builds business-ready Gold layer tables from clean Silver data.
- **general_ledger**: Union of all posting transactions with signed amounts
- **trial_balance**: Period-level account balances for P&L and Balance Sheet reporting

Always runs as a full overwrite — Gold tables are rebuilt from Silver each execution.

## Configuration

In [ ]:
spark.conf.set("spark.sql.ansi.enabled", "false")

In [ ]:
import os
import pandas as pd
from datetime import datetime
from pyspark.sql import functions as F
import logging
import traceback

In [ ]:
logger = logging.getLogger(__name__)

dbutils.widgets.text("run_id", "", "Job Run ID")

try:
    run_id = dbutils.widgets.get('run_id')
    if not run_id:
        run_id = f"manual_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
except Exception:
    run_id = f"manual_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

env = dbutils.widgets.get("env")
if env.lower() in ["prod", "p", "pr"]:
    CATALOG = 'analytics_lakehouse'
else:
    CATALOG = 'analytics_lakehouse_dev'

TARGET_SCHEMA = 'gold'
SOURCE_SCHEMA = 'silver'
QUERY_PATH = '../../queries/gold'

logger.info(f"Environment: {env}. Reading from {CATALOG}.{SOURCE_SCHEMA}, writing to {CATALOG}.{TARGET_SCHEMA}")
logger.info(f"Job Run ID: {run_id}")

start_time = datetime.now()
start_timestamp = spark.sql("SELECT current_timestamp()").collect()[0][0]
logger.info(f"Start Time: {start_timestamp}")

In [ ]:
# Gold tables in dependency order — general_ledger must be built before trial_balance
TABLES = [
    "general_ledger",
    "trial_balance",
]

## Helper Functions

In [ ]:
%run "/Workspace/Users/mike.asim1999@gmail.com/main/databricks-shared-utilities/functions/ensure_logging_table_exists"

In [ ]:
%run "/Workspace/Users/mike.asim1999@gmail.com/main/databricks-shared-utilities/functions/log_etl_run"

## Main ETL Function

In [ ]:
def main_etl(table_name):
    """
    Processes a single Gold table.
    
    Gold tables are always full overwrites built from Silver.
    SQL files are read from queries/gold/ and parameterized with CATALOG/schema names.
    """
    result = {"table": table_name, "success": False, "skipped": False}
    table_start = datetime.now()
    num_source = 0
    
    target_table = f"{CATALOG}.{TARGET_SCHEMA}.{table_name}"
    
    try:
        # Read SQL file
        sql_path = os.path.join(QUERY_PATH, f"{table_name}.sql")
        if not os.path.exists(sql_path):
            raise FileNotFoundError(f"SQL file not found: {sql_path}")
        
        with open(sql_path, 'r', encoding='utf-8') as f:
            raw_sql = f.read().strip()
        
        # Format SQL with parameters
        sql_query = raw_sql.format(
            CATALOG=CATALOG,
            SOURCE_SCHEMA=SOURCE_SCHEMA,
            TARGET_SCHEMA=TARGET_SCHEMA,
        )
        
        # Execute query
        logger.info(f"Executing {table_name} query")
        read_start = datetime.now()
        df = spark.sql(sql_query)
        read_duration = (datetime.now() - read_start).total_seconds()
        
        # Write as overwrite
        write_start = datetime.now()
        num_source = df.count()
        
        if num_source == 0:
            logger.warning(f"No rows returned for {table_name} — skipping write")
            result.update({
                "success": True, "skipped": True,
                "source_row_count": 0, "target_row_count": 0,
                "duration": (datetime.now() - table_start).total_seconds(),
            })
            return result
        
        df.write.mode("overwrite") \
            .option("overwriteSchema", "true") \
            .format("delta") \
            .saveAsTable(target_table)
        
        write_duration = (datetime.now() - write_start).total_seconds()
        duration = (datetime.now() - table_start).total_seconds()
        
        logger.info(f"{table_name}: {num_source:,} rows written in {duration:.2f}s")
        
        result.update({
            "success": True,
            "source_row_count": num_source,
            "target_row_count": num_source,
            "num_inserted": num_source,
            "num_updated": 0,
            "num_deleted": 0,
            "num_copied": 0,
            "duration": duration,
            "read_duration": read_duration,
            "write_duration": write_duration,
            "effective_mode": "overwrite",
        })
        
    except Exception as e:
        duration = (datetime.now() - table_start).total_seconds()
        result["error"] = str(e)
        result["success"] = False
        logger.error(f"Failed: {table_name}")
        logger.error(f"Error: {str(e)}")
        logger.error(traceback.format_exc())
    
    finally:
        result.setdefault("duration", (datetime.now() - table_start).total_seconds())
        result.setdefault("effective_mode", "error")
        log_run(
            catalog=CATALOG,
            schema_name=TARGET_SCHEMA,
            run_context={
                "run_id": run_id,
                "table_name": table_name,
                "log_datetime": datetime.now(),
                "status": "SKIPPED" if result.get("skipped") else ("SUCCESS" if result["success"] else "FAILED"),
                "load_type": result.get("effective_mode", "unknown"),
                "error_message": result.get("error"),
                "duration_seconds": result.get("duration", 0.0),
                "duration_minutes": result.get("duration", 0.0) / 60.0,
                "read_duration_seconds": result.get("read_duration", 0.0),
                "write_duration_seconds": result.get("write_duration", 0.0),
                "source_row_count": result.get("source_row_count"),
                "target_row_count": result.get("target_row_count"),
                "inserted": result.get("num_inserted"),
                "updated": result.get("num_updated"),
                "deleted": result.get("num_deleted"),
                "copied": result.get("num_copied"),
            },
        )
    
    return result

## Execute ETL

In [ ]:
def execute_etl():
    """Runs Gold ETL sequentially — order matters (general_ledger before trial_balance)."""
    job_start_time = datetime.now()
    results = []
    
    try:
        ensure_logging_table(CATALOG, TARGET_SCHEMA)
    except Exception as e:
        logger.error(f"Failed to create logging table: {e}")
        raise
    
    logger.info("Running Gold ETL sequentially")
    for table_name in TABLES:
        try:
            res = main_etl(table_name)
            results.append(res)
            
            # Stop if a table fails — downstream tables depend on it
            if not res.get("success"):
                logger.error(f"{table_name} failed — stopping pipeline")
                break
                
        except Exception as e:
            logger.error(f"{table_name} — CRITICAL FAILURE: {e}")
            results.append({"table": table_name, "success": False, "error": str(e)})
            break
    
    job_duration = (datetime.now() - job_start_time).total_seconds()
    successful = sum(1 for r in results if r.get("success") and not r.get("skipped"))
    skipped = sum(1 for r in results if r.get("skipped"))
    failed = sum(1 for r in results if not r.get("success"))
    
    logger.info(f"Run ID: {run_id}")
    logger.info(f"Job Duration: {job_duration:.2f}s ({job_duration/60:.2f}min)")
    logger.info(f"Tables: {len(results)} | Success: {successful} | Skipped: {skipped} | Failed: {failed}")
    
    return results


table_results = execute_etl()

## Summary

In [ ]:
summary_rows = []
for r in table_results:
    table_name = r.get("table", "Unknown")
    success = r.get("success", False)
    skipped = r.get("skipped", False)
    
    if skipped:
        status = "Skipped"
    elif success:
        status = "Success"
    else:
        status = "Failed"
    
    if success and not skipped:
        row = {
            "Table": table_name,
            "Status": status,
            "Rows": f"{r.get('source_row_count', 0):,}",
            "Duration (s)": f"{r.get('duration', 0.0):.2f}",
            "Error": "",
        }
    elif skipped:
        row = {
            "Table": table_name,
            "Status": status,
            "Rows": "0",
            "Duration (s)": f"{r.get('duration', 0.0):.2f}",
            "Error": "",
        }
    else:
        row = {
            "Table": table_name,
            "Status": status,
            "Rows": "-",
            "Duration (s)": f"{r.get('duration', 0.0):.2f}",
            "Error": r.get("error", "Unknown"),
        }
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
display(summary_df)